Question 1.1
Write a function that reads every speech file into a single frame-level table, with one row per frame: the identifier of the video, the fields of the file name that are needed, frame, timestamp, confidence and the 136 coordinates. Report its dimensions, and mark the rows of the four test actors. Reading only the required columns, and saving the table once in a binary format such as Parquet, avoids reading the CSV files again in every session.

In [ ]:
import os 
import pandas as pd
from pathlib import Path
import re

path = "../data"
DATA = Path(path)
useCols = ["frame","timestamp","confidence"]

for i in range(0,68):
    useCols.append(f"x_{i}")
    useCols.append(f"y_{i}")

testIndex = []
testActors = ["01","17","16","24"]

df = pd.DataFrame()
for root, dirs, files in os.walk(path):
    for n in files:
        p = re.split(r'[-.\s]+',n)
        
        if(p[1]) == "02" :
            continue
        if p[6] in testActors:
            testIndex.append(n)
        csv = pd.read_csv(DATA / n,usecols=useCols)
        csv.insert(1, "video", n)
        df = pd.concat([df,csv])

df.to_parquet("df.parquet.gzip", compression="gzip")
pd.read_parquet("df.parquet.gzip")       
        


Question 1.2
Build a video-level table of the development actors, with one row per video: actor, sex, emotion, intensity, statement, repetition and number of frames. Report the number of videos per emotion and per intensity, and the distribution of the number of frames per emotion.

In [39]:
import os 
import pandas as pd
from pathlib import Path
import re

path = "../data"
DATA = Path(path)


df = []
emotionLabels = ["Neutral", "Calm", "Happy", "Sad", "Angry", "Fearful", "Disgust", "Surprised"]
intensityLabels = ["Normal", "Strong"]

for root, dirs, files in os.walk(path):
    for n in files:
        p = re.split(r'[-.\s]+',n)
        
        if(p[1]) == "02" :
            continue
        
        
        csv = pd.read_csv(DATA / n,usecols=useCols)
        sex = 'F' if int(p[6])%2 == 0 else 'M'

        emotion = emotionLabels[int(p[2]) - 1]
        intensity = intensityLabels[int(p[3]) - 1]

        video = {"actor": p[6], "sex": sex, "emotion": emotion, "intensity" : intensity, "statement": p[4], "repetition": p[5], "number of frames" : len(csv)}
        df.append(video)
df = pd.DataFrame(df)
print(df)
    
        


     actor sex  emotion intensity statement repetition  number of frames
0       12   F    Happy    Strong        02         01               116
1       06   F    Happy    Strong        02         01               106
2       16   F      Sad    Strong        02         01               116
3       02   F  Disgust    Strong        02         02               121
4       02   F      Sad    Strong        02         01               111
...    ...  ..      ...       ...       ...        ...               ...
1435    13   M      Sad    Strong        01         01               100
1436    13   M  Disgust    Strong        01         02               111
1437    07   M      Sad    Strong        01         01               118
1438    23   M    Happy    Normal        01         01               100
1439    23   M    Happy    Strong        02         01                97

[1440 rows x 7 columns]
